# 01 · Adquisición y estandarización de datos

Carga de los tres datasets (ISIC, Stanford/MIDAS, Brasil/PAD-UFES-20), filtro por escala Fitzpatrick (III–VI) y estandarización a un esquema común de 8 columnas.

In [ ]:
# --- Configuración del entorno (portable: local / Colab) ---
import sys
from pathlib import Path

# Raíz del proyecto: dos niveles arriba de este notebook (Code/ -> raíz)
PROJECT_ROOT = Path().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import CFG  # noqa: E402  (rutas y constantes centralizadas)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos: {CFG.DATA_DIR}")

In [ ]:
# Libraries

import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
# Data path (ruta absoluta desde la configuración centralizada)
data_path = CFG.DATA_DIR

# Obteniendo los datos de ISIC

In [ ]:
df_ISIC = pd.read_csv(data_path / 'ISIC-Data/train.csv')
df_ISIC.head()

In [ ]:
df_ISIC["anatom_site_general_challenge"].value_counts()

In [ ]:
df_ISIC[
    (df_ISIC["anatom_site_general_challenge"] == "palms/soles")
    &
    (df_ISIC["target"] == 1)
].value_counts()

In [ ]:
df_ISIC.columns

## Número de diagnósticos por tipo

In [ ]:
df_ISIC["diagnosis"].value_counts()

In [ ]:
df_ISIC["benign_malignant"].value_counts()

# Realización con PROXY

En este caso no tenemos como tal una columna con un indicativo de Fitzpatrick (esta va desde Tipo 1 a Tipo VII), por tal razón, se va a utilizar un criterio que indica que las personas afectas por esta enfermedad, tienen tendencias a verse afectadas en las palmas y las plantas de los pies.

In [ ]:
# Filtro para melanomas en palmas o plantas
real_case = df_ISIC[
    (df_ISIC['anatom_site_general_challenge'] == 'palms/soles') &
    (df_ISIC["target"] == 1)
    ]
real_case = real_case.reset_index(drop= True)

In [ ]:
df_positivos = df_ISIC[(df_ISIC['anatom_site_general_challenge'] != 'palms/soles')].copy()
df_muestra = df_positivos.groupby('anatom_site_general_challenge').head(3)
df_muestra = df_muestra.reset_index(drop=True)

In [ ]:
df_ISIC_proxy = pd.concat([real_case, df_muestra], ignore_index= True)
df_ISIC_proxy.head()

In [ ]:
df_ISIC_proxy.shape

In [ ]:
df_ISIC_proxy.head()

## Obteniendo los datos de Stanford

In [ ]:
df_ddi = pd.read_csv(data_path / "midasmultimodalimagedatasetforaibasedskincancer/release_midas.csv", sep = ";")

df_ddi.head()

In [ ]:
df_ddi.shape

In [ ]:
df_ddi["midas_fitzpatrick"].unique()

In [ ]:
valores_de_piel = ['iii darker white skin', 'iv light brown skin', 'v brown skin', 'vi dark brown or black skin']

df_ddi = df_ddi[
                df_ddi["midas_fitzpatrick"].isin(valores_de_piel)
                ]

df_ddi.shape

In [ ]:
df_ddi.columns

In [ ]:
df_ddi.head()

## Obteniendo los datos de Brazil

In [ ]:
df_brazil = pd.read_csv(data_path / "zr7vgbcyr2-1/metadata.csv")

In [ ]:
df_brazil.head()

In [ ]:
df_brazil.columns

In [ ]:
df_brazil.info(memory_usage= 'deep')

In [ ]:
df_brazil["fitspatrick"].unique()

In [ ]:
val_braz = [3., 4., 5., 6.]

df_brazil = df_brazil[
            df_brazil["fitspatrick"].isin(val_braz)
        ]

df_brazil.shape

In [ ]:
df_brazil.head()

# Estandarización del Data Frame

In [ ]:
# Brasil
columnas_brasil = [
    'img_id',
    'patient_id',
    'age',
    'gender',
    'region',
    'diagnostic',
    'fitspatrick'
]
df_brazil = df_brazil[columnas_brasil]

# Stanford
columnas_stanford = [
    'midas_file_name',
    'midas_record_id',
    'midas_age',
    'midas_gender',
    'midas_location',
    'clinical_impression_1',
    'midas_fitzpatrick'
]
df_ddi = df_ddi[columnas_stanford]

# ISIC
columnas_isic = [
    'image_name',
    'patient_id',
    'age_approx',
    'sex',
    'anatom_site_general_challenge',
    'diagnosis'
]
df_ISIC_proxy = df_ISIC_proxy[columnas_isic]

In [ ]:
df_brazil.head()

In [ ]:
df_ddi.head()

In [ ]:
df_ISIC_proxy.head()

Aquí se debe de construir una variable para ISIC, dado que se ha realizado un `PROXY` para
determinar aquellos posibles casos que son cáncer. En este caso se va a utilizar *fitzpatrick*
como "0", este indicaría 'Desconocido'.

In [ ]:
df_ISIC_proxy["fitzpatrick"] = 0

# Estandarización y recodificación de cada variable

In [ ]:
df_ISIC_proxy.head()

In [ ]:
df_ISIC_proxy["anatom_site_general_challenge"].unique()

In [ ]:
df_brazil.head()

In [ ]:
df_brazil["region"].unique()

In [ ]:
df_ISIC_proxy["anatom_site_general_challenge"].unique()

***

Configuración de Location

### ISIC basados

Stanford:

In [ ]:
stanford_change = pd.read_csv(data_path / "zr7vgbcyr2-1/ubicaciones_unicas.csv", sep= ";")

In [ ]:
mapeo_ubicaciones = dict(zip(
    df_ddi["midas_location"].str.lower(),
    stanford_change["Nueva Categoria General"]
))

df_ddi["midas_location"] = df_ddi["midas_location"].str.lower().map(mapeo_ubicaciones)

In [ ]:
df_ddi["midas_location"].unique()

Brasil:

In [ ]:
brasil_change = {
    'NECK': 'head/neck',
    'FACE': 'head/neck',
    'FOREARM': 'upper extremity',
    'BACK': 'torso',
    'HAND': 'upper extremity',
    'CHEST': 'torso',
    'NOSE': 'head/neck',
    'EAR': 'head/neck',
    'SCALP': 'head/neck',
    'ARM': 'upper extremity',
    'THIGH': 'lower extremity',
    'ABDOMEN': 'torso',
    'FOOT': 'lower extremity',
    'LIP': 'head/neck'
}

In [ ]:
df_brazil["region"] = df_brazil["region"].replace(brasil_change)

In [ ]:
df_brazil["region"].unique()

***

# Configuración general para un Pipeline de preparamiento de los datos

In [ ]:
columnas_estandarizadas = [
    'image_id',
    'patient_id',
    'age',
    'sex',
    'location',
    'diagnosis',
    'fitzpatrick_skin_type',
    'source'
]

In [ ]:
def procesar_isic(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia y estandariza el DataFrame de ISIC."""
    # Mapeo de nombres de columnas
    mapeo_columnas = {
        'image_name': 'image_id',
        'age_approx': 'age',
        'anatom_site_general_challenge': 'location'
    }
    df = df.rename(columns=mapeo_columnas)
    
    # ISIC no tiene tipo de piel, ahora será 0
    df['fitzpatrick_skin_type'] = 0
    
    # Recode para 'diagnosis' - Lo saco de Brazil este formato
    mapeo_diag = {
    'nevus': "NEV",
    'melanoma': "MEL",
    'unknown' : "otro"
    }
    df["diagnosis"] = df["diagnosis"].str.lower().map(mapeo_diag)
    df['source'] = 'isic'
    
    # Seleccionamos y reordenamos las columnas para que coincidan con el estándar
    return df[columnas_estandarizadas]

def procesar_stanford(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia y estandariza el DataFrame de Stanford (MIDAS)."""
    mapeo_columnas = {
        'midas_file_name': 'image_id',
        'midas_record_id': 'patient_id',
        'midas_gender': 'sex',
        'midas_age': 'age',
        'midas_location': 'location',
        'clinical_impression_1': 'diagnosis',
        'midas_fitzpatrick': 'fitzpatrick_skin_type'
    }
    df = df.rename(columns=mapeo_columnas)
    
    # --- Recodificación de Valores ---
    # Estandarizar sexo a minúsculas
    df['sex'] = df['sex'].str.lower()
    
    # --- Recodificación de Stanford ---
    mapeo_a_abreviaturas = {
    '11-malignant-melanoma': 'MEL',
    '7-malignant-bcc': 'BCC',
    '8-malignant-scc': 'SCC',
    '10-malignant-ak': 'ACK',
    '1-benign-melanocytic nevus': 'NEV',
    '2-benign-seborrheic keratosis': 'SEK',
    '4-benign-dermatofibroma': 'otro',
    '14-other-non-neoplastic/inflammatory/infectious': 'otro',
    '6-benign-other': 'otro',
    '12-malignant-other': 'otro',
    '5-benign-hemangioma': 'otro',
    '3-benign-fibrous papule': 'otro'
    }

    df["diagnosis"] = df['diagnosis'].str.lower().map(mapeo_a_abreviaturas)

    # Convertir Fitzpatrick de números romanos en string a números enteros
    fitzpatrick_map = {'iii darker white skin': 3, 'iv light brown skin': 4, 'v brown skin': 5,
                        'vi dark brown or black skin': 6}
    df['fitzpatrick_skin_type'] = df['fitzpatrick_skin_type'].str.lower().map(fitzpatrick_map)
    
    df['source'] = 'stanford'
    
    return df[columnas_estandarizadas]

def procesar_brasil(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia y estandariza el DataFrame de Brasil (PAD-UFES-20)."""
    mapeo_columnas = {
        'img_id': 'image_id',
        'gender': 'sex',
        'region': 'location',
        'diagnostic': 'diagnosis',
        'fitspatrick': 'fitzpatrick_skin_type'
    }
    df = df.rename(columns=mapeo_columnas)
    
    # --- Recodificación de Valores ---
    df['sex'] = df['sex'].str.lower()
    df['location'] = df['location'].str.lower()
    df['diagnosis'] = df['diagnosis'].str.lower()
    df['source'] = 'brasil'
    
    return df[columnas_estandarizadas]

# FIN
***